# IndoBERT Token Classifier (Colab)

Notebook ini untuk inferensi label token pantun modern dengan skema label BIO (`B/I-ASG`, `B/I-SKT`, `B/I-DSM`, `B/I-TYP`, `O`) memakai model IndoBERT yang disimpan di Google Drive.

Sesuai dokumentasi pipeline:
- `B/I-SKT` -> Rumus A (fonemisasi alfabetis)
- `B/I-DSM` / `B/I-TYP` -> Rumus B & C
- `B/I-ASG` -> failsafe G2P asing
- `O` -> Cascaded CRF 1 (G2P kata baku)


In [ ]:
!pip -q install transformers sentencepiece accelerate g2p-id g2p-en nltk symspellpy sklearn-crfsuite joblib pyphen

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
# UBAH sesuai lokasi folder model fine-tuned Anda di Google Drive.
# Folder ini minimal berisi: config.json, pytorch_model.bin/model.safetensors,
# tokenizer.json atau vocab.txt + tokenizer config.
MODEL_DIR = '/content/drive/MyDrive/Models/indobert-pantun-ner'  # <- ganti path ini

# Kamus kata dasar (1 kata per baris) untuk SymSpell recover typo B/I-TYP.
# Silakan taruh file ini di Google Drive Anda.
KATA_DASAR_PATH = '/content/drive/MyDrive/kata-dasar.txt'  # <- ganti bila perlu

# Model CRF layer-3 (syllabifier B/I dari deret fonem hasil G2P).
CRF_BI_MODEL_PATH = '/content/drive/MyDrive/Models/crf_layer3_bi.joblib'  # <- ganti bila perlu


In [ ]:
import os
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id
print('Labels:', id2label)

# Preload CRF BI di bagian atas agar edit logic helper tidak memicu reload model.
crf_bi_model = None
crf_target = str(globals().get('CRF_BI_MODEL_PATH', '')).strip()
if crf_target and os.path.exists(crf_target):
    crf_bi_model = joblib.load(crf_target)
    print(f'[CRF3] Loaded model: {crf_target}')
else:
    print(f'[CRF3] Model belum dimuat (path kosong/tidak ditemukan): {crf_target}')

In [ ]:
# Setup/training SymSpell dari kamus kata dasar (jalankan sebelum cell helper)
import os
from symspellpy import SymSpell


def _resolve_kata_dasar_path() -> str:
    configured = globals().get('KATA_DASAR_PATH', '/content/drive/MyDrive/kata-dasar.txt')
    return str(configured).strip()


def build_symspell_from_kata_dasar(dict_path: str, max_edit_distance: int = 2):
    if not dict_path or not os.path.exists(dict_path):
        print(f'[SymSpell] Dictionary tidak ditemukan: {dict_path}. Recover typo B/I-TYP dinonaktifkan.')
        return None

    symspell = SymSpell(max_dictionary_edit_distance=max_edit_distance, prefix_length=7)
    loaded = 0

    with open(dict_path, 'r', encoding='utf-8') as f:
        for raw in f:
            term = raw.strip().lower()
            if not term or not term.isalpha():
                continue
            symspell.create_dictionary_entry(term, 1)
            loaded += 1

    if loaded == 0:
        print(f'[SymSpell] File kamus kosong/tidak valid: {dict_path}. Recover typo B/I-TYP dinonaktifkan.')
        return None

    print(f'[SymSpell] Loaded {loaded} kata dari: {dict_path}')
    return symspell


def build_dsm_skeleton_index(dict_path: str):
    """Index kata dasar berdasarkan skeleton konsonan (hapus vokal)."""
    if not dict_path or not os.path.exists(dict_path):
        print(f'[DSM-SKEL] Dictionary tidak ditemukan: {dict_path}. Recover DSM dinonaktifkan.')
        return {}

    vowels = set('aiueo')
    index = {}
    loaded = 0
    with open(dict_path, 'r', encoding='utf-8') as f:
        for raw in f:
            term = raw.strip().lower()
            if not term or not term.isalpha():
                continue
            skeleton = ''.join(ch for ch in term if ch not in vowels)
            if not skeleton:
                continue
            bucket = index.setdefault(skeleton, [])
            if term not in bucket:
                bucket.append(term)
            loaded += 1

    print(f'[DSM-SKEL] Loaded {loaded} kata, {len(index)} skeleton unik dari: {dict_path}')
    return index


symspell_model = build_symspell_from_kata_dasar(_resolve_kata_dasar_path())

dsm_skeleton_index = build_dsm_skeleton_index(_resolve_kata_dasar_path())

In [ ]:
def _clean_input_words(text: str) -> list[str]:
    words = []
    for raw in str(text).strip().split():
        cleaned = re.sub(r"^[^\\w]+|[^\\w]+$", "", raw)
        if cleaned:
            words.append(cleaned)
    return words


def predict_token_labels(text: str):
    """Inferensi token-level label + confidence dengan penyelarasan wordpiece -> token asli."""
    words = _clean_input_words(text)
    if not words:
        return []
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        padding=False
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_ids = torch.argmax(logits, dim=-1)[0].tolist()

    word_ids = tokenizer(words, is_split_into_words=True).word_ids()

    token_rows = []
    current_word = None
    for idx, widx in enumerate(word_ids):
        if widx is None:
            continue
        if widx != current_word:
            current_word = widx
            label = id2label[pred_ids[idx]]

            top2 = torch.topk(probs[idx], k=min(2, probs[idx].shape[-1]))
            confidence = float(top2.values[0].item())
            second_best = float(top2.values[1].item()) if top2.values.shape[0] > 1 else 0.0
            margin = confidence - second_best

            token_rows.append(
                {
                    'token': words[widx],
                    'label': label,
                    'confidence': confidence,
                    'margin': margin,
                }
            )
    return token_rows


LABELS_ASING = {"B-ASG", "I-ASG"}
LABELS_SINGKATAN = {"B-SKT", "I-SKT"}
LABELS_DISEMVOWEL = {"B-DSM", "I-DSM"}
LABELS_TYPO = {"B-TYP", "I-TYP"}
LABELS_VALID = {"O", *LABELS_ASING, *LABELS_SINGKATAN, *LABELS_DISEMVOWEL, *LABELS_TYPO}


def route_by_label(label: str) -> str:
    """Routing sesuai dokumentasi pipeline dan skema label BIO IndoBERT."""
    norm = label.upper().strip()
    if norm not in LABELS_VALID:
        raise ValueError(f'Label tidak dikenali: {label}')

    if norm in LABELS_SINGKATAN:
        return 'RUMUS_A_FONEMISASI_ALFABETIS'
    if norm in LABELS_DISEMVOWEL or norm in LABELS_TYPO:
        return 'RUMUS_B_C_FONEMISASI_DISEMVOWELLING'
    if norm in LABELS_ASING:
        return 'FAILSAFE_G2P_ASING'
    return 'CRF1_G2P_KATA_BAKU'


def is_likely_disemvowelling(token: str) -> bool:
    """Heuristik robust untuk mendeteksi pola disemvowelling tanpa kamus hardcoded."""
    t = token.lower().strip()
    if not t.isalpha() or len(t) < 3:
        return False

    vowels = set('aiueo')
    vowel_count = sum(ch in vowels for ch in t)
    consonant_ratio = (len(t) - vowel_count) / len(t)

    max_cons_run = 0
    curr_run = 0
    for ch in t:
        if ch in vowels:
            curr_run = 0
        else:
            curr_run += 1
            max_cons_run = max(max_cons_run, curr_run)

    consonant_clusters = (
        'ng', 'ny', 'kh', 'sy', 'tr', 'br', 'gr', 'pr', 'kr', 'sk', 'st', 'nd', 'mb'
    )
    cluster_hits = sum(1 for c in consonant_clusters if c in t)

    score = 0
    if vowel_count == 0:
        score += 3
    elif vowel_count == 1 and len(t) >= 4:
        score += 2
    if consonant_ratio >= 0.70:
        score += 2
    if max_cons_run >= 3:
        score += 2
    if cluster_hits >= 1:
        score += 1
    if len(t) >= 4 and t[0] not in vowels and t[-1] not in vowels and vowel_count <= 1:
        score += 1

    # Ambang 4 adalah heuristik operasional: dipilih agar sinyal disemvowelling
    # (vokal minim + run konsonan/cluster) harus muncul bersamaan, bukan single cue.
    # Ini bukan hasil hyperparameter tuning formal lintas-validation set.
    return score >= 4


def override_typo_to_dsm(token: str, label: str) -> str:
    norm = label.upper().strip()
    if norm in LABELS_TYPO and is_likely_disemvowelling(token):
        recover_fn = globals().get('symspell_recover_typo')
        if callable(recover_fn):
            recovered = recover_fn(token)
            if recovered:
                return norm
        return norm.replace('TYP', 'DSM')
    return norm


def predict_with_routing(text: str):
    labeled = predict_token_labels(text)
    rows = []
    for item in labeled:
        tok = item['token']
        lab = item['label']
        token_meta = {
            'confidence': item.get('confidence'),
            'margin': item.get('margin'),
        }

        final_label = override_typo_to_dsm(tok, lab)
        switch = acoustic_switchboard(tok, final_label, token_meta=token_meta)
        units_now = phoneme_to_units(switch['phoneme'])

        bi_fn = globals().get('predict_bi_from_phoneme')
        bi_pattern = bi_fn(switch['phoneme']) if callable(bi_fn) else None
        bi_source = 'CRF3_BI' if bi_pattern else None

        if str(switch.get('switch_route', '')).startswith('ASG_'):
            refine_fn = globals().get('refine_asg_bi_with_pyphen')
            if callable(refine_fn):
                refined_bi, refined_source = refine_fn(tok, units_now, bi_pattern, switch.get('switch_route'))
                if refined_bi:
                    bi_pattern = refined_bi
                    bi_source = refined_source or bi_source

        rows.append(
            {
                'token': tok,
                'raw_label': lab,
                'label': final_label,
                'route': route_by_label(final_label),
                'model_confidence': token_meta['confidence'],
                'model_margin': token_meta['margin'],
                'switch_route': switch['switch_route'],
                'switch_input': switch['switch_input'],
                'phoneme': switch['phoneme'],
                'bi_pattern': bi_pattern,
                'bi_source': bi_source,
                'bc_info': switch.get('bc_info')
            }
        )
    return rows


In [ ]:
# Helper sederhana Rumus A (singkatan)
from functools import lru_cache
import os
import re
import nltk
import joblib
import pyphen
from symspellpy import Verbosity
from g2p_id import G2P as G2P_ID
from g2p_en import G2p as G2P_EN


def _ensure_nltk_resource(resource_path: str, package_name: str):
    try:
        nltk.data.find(resource_path)
    except LookupError:
        nltk.download(package_name, quiet=True)


_ensure_nltk_resource('corpora/cmudict', 'cmudict')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng')
_ensure_nltk_resource('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger')

g2p_id_model = G2P_ID()
g2p_en_model = G2P_EN()

try:
    pyphen_en_dict = pyphen.Pyphen(lang='en_US')
except Exception:
    pyphen_en_dict = None

ALPHABET_EJA_MAP = {
    "a": "a", "b": "be", "c": "ce", "d": "de", "e": "e", "f": "ef", "g": "ge",
    "h": "ha", "i": "i", "j": "je", "k": "ka", "l": "el", "m": "em", "n": "en",
    "o": "o", "p": "pe", "q": "ki", "r": "er", "s": "es", "t": "te", "u": "u",
    "v": "ve", "w": "we", "x": "eks", "y": "ye", "z": "zet"
}

# Konversi simbol fonem Inggris (ARPAbet) ke fonetik internasional (IPA)
ARPABET_TO_IPA = {
    "AA": "ɑ", "AE": "æ", "AH": "ʌ", "AO": "ɔ", "AW": "aʊ", "AY": "aɪ",
    "B": "b", "CH": "tʃ", "D": "d", "DH": "ð", "EH": "ɛ", "ER": "ɝ",
    "EY": "eɪ", "F": "f", "G": "ɡ", "HH": "h", "IH": "ɪ", "IY": "i",
    "JH": "dʒ", "K": "k", "L": "l", "M": "m", "NG": "ŋ", "OW": "oʊ",
    "OY": "ɔɪ", "P": "p", "R": "r", "S": "s", "SH": "ʃ", "T": "t",
    "TH": "θ", "UH": "ʊ", "UW": "u", "V": "v", "W": "w", "Y": "j",
    "Z": "z", "ZH": "ʒ"
}




CRF3_VOWELS = {"a", "i", "u", "e", "o", "ə", "ɛ", "ɪ", "ʊ", "ɔ", "æ", "ɐ", "ɒ", "ɑ", "ʌ", "ɝ"}
crf_bi_model = globals().get('crf_bi_model')


def load_crf_bi_model(model_path=None):
    """Load model CRF layer-3 (.joblib)."""
    global crf_bi_model
    target = str(model_path or globals().get('CRF_BI_MODEL_PATH', '')).strip()
    if not target:
        print('[CRF3] Path model kosong. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None
    if not os.path.exists(target):
        print(f'[CRF3] Model tidak ditemukan: {target}. Prediksi BI dinonaktifkan.')
        crf_bi_model = None
        return None

    crf_bi_model = joblib.load(target)
    print(f'[CRF3] Loaded model: {target}')
    return crf_bi_model


CRF3_DIPHTHONGS = ("eɪ", "aɪ", "aʊ", "ɔɪ", "oʊ")


def phoneme_to_units(phoneme: str) -> list[str]:
    text = str(phoneme).strip()
    if not text:
        return []
    if ' ' in text:
        return [u for u in text.split() if u]

    chars = [ch for ch in text if not ch.isspace()]
    units: list[str] = []
    i = 0
    while i < len(chars):
        if i + 1 < len(chars):
            pair = chars[i] + chars[i + 1]
            if pair in CRF3_DIPHTHONGS:
                units.append(pair)
                i += 2
                continue
        units.append(chars[i])
        i += 1
    return units


def _crf3_is_vowel(phone: str) -> bool:
    return phone in CRF3_VOWELS


def _crf3_phone_features(seq: list[str], i: int) -> dict:
    cur = seq[i]
    prev = seq[i - 1] if i > 0 else '<BOS>'
    nxt = seq[i + 1] if i < len(seq) - 1 else '<EOS>'
    return {
        'bias': 1.0,
        'phone': cur,
        'prev_phone': prev,
        'next_phone': nxt,
        'bigram_prev_cur': f'{prev}+{cur}',
        'bigram_cur_next': f'{cur}+{nxt}',
        'is_vowel': _crf3_is_vowel(cur),
        'prev_is_vowel': False if prev == '<BOS>' else _crf3_is_vowel(prev),
        'next_is_vowel': False if nxt == '<EOS>' else _crf3_is_vowel(nxt),
        'pos': i,
        'rev_pos': len(seq) - 1 - i,
        'len_seq': len(seq),
        'bos': i == 0,
        'eos': i == len(seq) - 1,
    }


def predict_bi_from_phoneme(phoneme: str, model=None):
    mdl = model if model is not None else globals().get('crf_bi_model')
    if mdl is None:
        return None

    units = phoneme_to_units(phoneme)
    if not units:
        return None

    feats = [_crf3_phone_features(units, i) for i in range(len(units))]
    pred = mdl.predict_single(feats)
    return ''.join(pred)

def _count_b_from_bi(bi_pattern: str | None) -> int:
    return sum(1 for ch in (bi_pattern or '') if ch == 'B')


def _pyphen_syllable_count(token: str) -> int | None:
    dic = globals().get('pyphen_en_dict')
    if dic is None:
        return None
    t = normalize_typo_token(token)
    if not t:
        return None
    inserted = dic.inserted(t)
    if not inserted:
        return None
    return inserted.count('-') + 1


def _orthographic_vowel_group_count(token: str) -> int:
    t = normalize_typo_token(token)
    if not t:
        return 0
    return len(re.findall(r'[aiueo]+', t))


def is_localized_asg_token(token: str) -> bool:
    t = normalize_typo_token(token)
    if len(t) < 7:
        return False

    strong_english_markers = ('th', 'sh', 'ch', 'ph', 'ck', 'qu', 'tion', 'sion', 'ing')
    if any(m in t for m in strong_english_markers):
        return False

    vowel_letters = sum(ch in 'aiueo' for ch in t)
    vowel_groups = len(re.findall(r'[aiueo]+', t))
    return vowel_letters >= 4 and vowel_groups >= 3


def build_bi_from_target_syllables(units: list[str], target_syllables: int) -> str | None:
    if not units:
        return None

    n = len(units)
    target = max(1, min(int(target_syllables), n))
    if target == 1:
        return 'B' + ('I' * (n - 1))

    labels = ['I'] * n
    labels[0] = 'B'

    candidate = [i for i in range(1, n) if _is_vowel_unit_for_nucleus(units[i])]
    if len(candidate) < target - 1:
        candidate = sorted(set(candidate + list(range(1, n))))

    if not candidate:
        return 'B' + ('I' * (n - 1))

    k = target - 1
    if k >= len(candidate):
        chosen = candidate
    else:
        chosen = []
        for j in range(k):
            idx = round(j * (len(candidate) - 1) / (k - 1)) if k > 1 else 0
            chosen.append(candidate[idx])
        chosen = sorted(set(chosen))

    for idx in chosen:
        labels[idx] = 'B'

    return ''.join(labels)


def refine_asg_bi_with_pyphen(
    token: str,
    units: list[str],
    bi_pattern: str | None,
    switch_route: str | None = None,
) -> tuple[str | None, str | None]:
    if not units:
        return bi_pattern, None

    pyphen_count = _pyphen_syllable_count(token)
    target_count = pyphen_count

    # Fallback untuk kata serapan/kolokial yang sering undercount di pyphen EN
    # (contoh: vaksin) agar tidak terkunci menjadi 1 suku kata.
    if switch_route == 'ASG_G2P_EN_IPA' and (target_count is None or target_count <= 1):
        vowel_groups = _orthographic_vowel_group_count(token)
        if vowel_groups >= 2:
            target_count = vowel_groups

    if target_count is None:
        return bi_pattern, None

    crf_count = _count_b_from_bi(bi_pattern)
    collapsed = crf_count <= 1 and len(units) >= 4
    strongly_mismatch = crf_count > 0 and abs(crf_count - target_count) >= 2
    mild_underseg_en = (
        switch_route == 'ASG_G2P_EN_IPA'
        and crf_count > 0
        and target_count >= 3
        and crf_count < target_count
        and (target_count - crf_count) == 1
    )

    if collapsed or strongly_mismatch or mild_underseg_en:
        rebuilt = build_bi_from_target_syllables(units, target_count)
        if rebuilt:
            return rebuilt, 'ASG_PYPHEN_BI'

    return bi_pattern, None


def choose_asg_phoneme(token: str) -> dict:
    cleaned = normalize_typo_token(token)
    if is_localized_asg_token(cleaned):
        return {
            'switch_route': 'ASG_G2P_ID_LOCALIZED',
            'switch_input': cleaned,
            'phoneme': g2p_id_phoneme(cleaned),
        }

    return {
        'switch_route': 'ASG_G2P_EN_IPA',
        'switch_input': cleaned,
        'phoneme': g2p_foreign_phoneme(cleaned),
    }


def singkatan_to_ejaan(word: str) -> str:
    """dpr -> de pe er"""
    clean = "".join(ch for ch in word.lower() if ch.isalpha())
    return " ".join(ALPHABET_EJA_MAP.get(ch, ch) for ch in clean)


def arpabet_to_ipa(arpabet_tokens: list[str]) -> str:
    ipa_tokens = []
    for token in arpabet_tokens:
        base = re.sub(r"\d", "", token).upper()
        if not base:
            continue
        ipa_tokens.append(ARPABET_TO_IPA.get(base, base.lower()))
    return "".join(ipa_tokens)


def _coerce_g2p_output_to_str(raw) -> str:
    if isinstance(raw, tuple):
        return _coerce_g2p_output_to_str(raw[0])
    if isinstance(raw, list):
        if raw and isinstance(raw[0], list):
            raw = [ph for sub in raw for ph in sub if ph]
        return " ".join(str(part) for part in raw if part)
    return str(raw).strip()


@lru_cache(maxsize=8192)
def g2p_id_phoneme(word: str) -> str:
    if not word:
        return ""
    try:
        res = g2p_id_model.to_phoneme(word)
    except AttributeError:
        try:
            res = g2p_id_model(word)
        except Exception:
            res = g2p_id_model.convert(word)
    return _coerce_g2p_output_to_str(res).replace(" ", "")


@lru_cache(maxsize=8192)
def g2p_foreign_phoneme(word: str) -> str:
    if not word:
        return ""
    raw = g2p_en_model(word)
    phones = [p for p in raw if p and p != " " and p != "<unk>"]
    ipa = arpabet_to_ipa(phones)
    return ipa if ipa else g2p_id_phoneme(word)


def rumus_a_singkatan_simple(word: str) -> str:
    """Rumus A: singkatan -> ejaan alfabet -> fonem Indonesia (g2p-id)."""
    ejaan = singkatan_to_ejaan(word)
    if not ejaan:
        return ""
    return g2p_id_phoneme(ejaan)


# -------------------------------
# Rumus B & C + typo handling (tanpa hardcoded kata)
# -------------------------------

VOWELS = set('aiueo')
COMMON_DIGRAPHS = ("ng", "ny", "kh", "sy", "br", "tr", "kr", "pr", "gr", "dr", "st", "sk")


def normalize_typo_token(token: str) -> str:
    """Normalisasi typo berbasis bentuk: lowercase, alfabet-only, kompres pengulangan huruf."""
    t = token.lower().strip()
    if not t:
        return t
    t = "".join(ch for ch in t if ch.isalpha())
    if not t:
        return t
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t


@lru_cache(maxsize=8192)
def symspell_recover_typo(
    token: str,
    allow_no_vowel: bool = False,
    max_edit_distance: int = 2,
    max_len_delta: int = 3,
) -> dict | None:
    """Recover typo ringan; return None jika tidak yakin."""
    model = globals().get('symspell_model')
    if model is None:
        return None

    normalized = normalize_typo_token(token)
    if not normalized:
        return None

    if (not allow_no_vowel) and all(ch not in VOWELS for ch in normalized):
        return None

    suggestions = model.lookup(
        normalized,
        Verbosity.TOP,
        max_edit_distance=max_edit_distance,
        include_unknown=True,
        transfer_casing=False,
    )
    if not suggestions:
        return None

    top = suggestions[0]
    recovered = top.term.lower().strip()
    if not recovered or recovered == normalized:
        return None

    allowed_dist = 1 if len(normalized) <= 5 else 2
    allowed_dist = min(allowed_dist, max_edit_distance)
    if top.distance > allowed_dist:
        return None
    if abs(len(recovered) - len(normalized)) > max_len_delta:
        return None

    return {
        'original': normalized,
        'recovered': recovered,
        'distance': int(top.distance),
        'count': int(top.count),
    }


def _strip_vowels(t: str) -> str:
    return "".join(ch for ch in t if ch not in VOWELS)


def _levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            ins = curr[j - 1] + 1
            delete = prev[j] + 1
            sub = prev[j - 1] + (0 if ca == cb else 1)
            curr.append(min(ins, delete, sub))
        prev = curr
    return prev[-1]


def dsm_recover_from_skeleton(token: str) -> dict | None:
    """Recover DSM via skeleton konsonan kamus kata dasar; ambigu => None."""
    index = globals().get('dsm_skeleton_index') or {}
    if not index:
        return None

    normalized = normalize_typo_token(token)
    if len(normalized) < 3:
        return None

    skeleton = _strip_vowels(normalized)
    if not skeleton:
        return None

    candidates = index.get(skeleton, [])
    if not candidates:
        return None

    token_vowels = sum(ch in VOWELS for ch in normalized)
    scored = []
    for cand in candidates:
        cand_vowels = sum(ch in VOWELS for ch in cand)
        score = (
            _levenshtein(normalized, cand),
            abs(len(cand) - len(normalized)),
            -(cand_vowels - token_vowels),
            cand,
        )
        scored.append((score, cand))

    scored.sort(key=lambda x: x[0])
    best_score, best_cand = scored[0]

    if len(scored) > 1:
        second_score, _ = scored[1]
        if second_score[:3] == best_score[:3]:
            return None

    dist, len_gap, neg_vgain, _ = best_score
    vowel_gain = -neg_vgain
    if dist > 3:
        return None
    if len_gap > 5:
        return None
    if token_vowels == 0 and vowel_gain <= 0:
        return None

    return {
        'original': normalized,
        'recovered': best_cand,
        'skeleton': skeleton,
        'distance': int(dist),
        'len_gap': int(len_gap),
        'vowel_gain': int(vowel_gain),
        'candidate_count': len(candidates),
    }


def _max_consonant_run(t: str) -> int:
    best = 0
    run = 0
    for ch in t:
        if ch in VOWELS:
            run = 0
        else:
            run += 1
            best = max(best, run)
    return best


def _has_common_cluster(t: str) -> bool:
    return any(dg in t for dg in COMMON_DIGRAPHS)


def _has_reversed_common_cluster(t: str) -> bool:
    reversed_digraphs = {dg[::-1] for dg in COMMON_DIGRAPHS if len(dg) == 2}
    return any(rdg in t for rdg in reversed_digraphs)


def _is_content_like_disemvowelling(t: str) -> bool:
    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    if vowel_count == 0:
        return True
    if vowel_count <= 1 and len(t) >= 4:
        return True
    if cons_run >= 3:
        return True
    if cons_run >= 2 and _has_common_cluster(t):
        return True
    return False


def is_low_confidence_label(token_meta: dict | None, conf_threshold: float = 0.80, margin_threshold: float = 0.45) -> bool:
    # conf=0.80 dan margin=0.45 dipakai sebagai gate konservatif untuk memicu
    # recover typo hanya saat model cukup ragu. Nilai ini berasal dari kalibrasi
    # empiris pada data pengembangan internal, bukan grid-search formal.
    if not token_meta:
        return False
    confidence = token_meta.get('confidence')
    margin = token_meta.get('margin')
    if confidence is None or margin is None:
        return False
    return float(confidence) <= conf_threshold or float(margin) <= margin_threshold


def should_attempt_o_recover(token: str, token_meta: dict | None = None) -> dict:
    normalized = normalize_typo_token(token)
    if len(normalized) < 3:
        return {'suspicious': False, 'normalized': normalized}

    vowel_count = sum(ch in VOWELS for ch in normalized)
    cons_run = _max_consonant_run(normalized)
    has_common_cluster = _has_common_cluster(normalized)
    has_reversed_cluster = _has_reversed_common_cluster(normalized)
    low_conf = is_low_confidence_label(token_meta)

    severe_no_vowel = vowel_count == 0 and cons_run >= 3
    sparse_vowel = len(normalized) >= 5 and vowel_count <= 1
    reversed_cluster_suspicious = has_reversed_cluster and (low_conf or sparse_vowel)
    suspicious = cons_run >= 2 and (not has_common_cluster) and (low_conf or severe_no_vowel or reversed_cluster_suspicious)
    return {
        'suspicious': suspicious,
        'normalized': normalized,
        'vowel_count': vowel_count,
        'cons_run': cons_run,
        'has_common_cluster': has_common_cluster,
        'has_reversed_cluster': has_reversed_cluster,
        'sparse_vowel': sparse_vowel,
        'severe_no_vowel': severe_no_vowel,
        'low_confidence': low_conf,
    }


def should_attempt_typo_recover(token: str, token_meta: dict | None = None) -> dict:
    normalized = normalize_typo_token(token)
    if len(normalized) < 3:
        return {
            'attempt_recover': False,
            'use_bc': False,
            'normalized': normalized,
        }

    vowel_count = sum(ch in VOWELS for ch in normalized)
    cons_run = _max_consonant_run(normalized)
    has_reversed_cluster = _has_reversed_common_cluster(normalized)
    disemvowel_like = _is_content_like_disemvowelling(normalized)
    sparse_vowel = len(normalized) >= 4 and vowel_count <= 1
    low_conf = is_low_confidence_label(token_meta)

    # Untuk label TYP, recovery dan BC hanya dipakai bila ada sinyal typo kuat.
    # Ini menghindari kata colloquial yang masih wajar (mis. ntar) dari koreksi agresif.
    severe_no_vowel = (vowel_count == 0 and cons_run >= 3)
    typo_strong_signal = has_reversed_cluster or low_conf or severe_no_vowel
    attempt_recover = disemvowel_like and typo_strong_signal
    use_bc = disemvowel_like and typo_strong_signal

    return {
        'attempt_recover': attempt_recover,
        'use_bc': use_bc,
        'normalized': normalized,
        'vowel_count': vowel_count,
        'cons_run': cons_run,
        'has_reversed_cluster': has_reversed_cluster,
        'disemvowel_like': disemvowel_like,
        'sparse_vowel': sparse_vowel,
        'severe_no_vowel': severe_no_vowel,
        'low_confidence': low_conf,
    }


def rumus_c_clamping(word: str, force_two_syllable: bool = False) -> str:
    """Rumus C: clamping panjang kata; opsional dipaksa 2 unit untuk typo pendek."""
    t = word.lower().strip()
    if force_two_syllable:
        return "vone" if t and t[-1] in VOWELS else "vonem"
    if len(t) <= 2:
        return "von"
    if len(t) == 3:
        return "vone" if t and t[-1] in VOWELS else "vonem"
    return "vone" if t and t[-1] in VOWELS else "vonem"


def rumus_b_phonotactic_mapping(word: str) -> str:
    """Rumus B: estimasi struktur fonotaktik untuk kata konten tanpa vokal."""
    t = "".join(ch for ch in word.lower() if ch.isalpha())
    if not t:
        return "von"

    vowel_count = sum(ch in VOWELS for ch in t)
    cons_run = _max_consonant_run(t)
    has_cluster = _has_common_cluster(t)

    if vowel_count == 0:
        sonorants = set('rlmnwy')
        if len(t) == 3 and any(ch in sonorants for ch in t):
            return "vonemi"
        if len(t) <= 4:
            syll_est = 2
        elif has_cluster or cons_run >= 4:
            syll_est = 3
        else:
            syll_est = 2
        patterns = {2: "vonem", 3: "vonemis", 4: "vonemisa"}
        return patterns[syll_est]

    compact = t
    for dg in COMMON_DIGRAPHS:
        compact = compact.replace(dg, "X")

    base_units = len(compact)
    has_coda = t[-1] in "bcdfghjklmnpqrstvwxyz"
    if has_coda and base_units > 1:
        base_units -= 1

    syll_est = max(2, min(4, (base_units + 1) // 2))

    patterns = {
        2: "vonem",
        3: "vonemis",
        4: "vonemisa",
    }
    return patterns[syll_est]


def rumus_bc_for_dsm_typo(token: str, label: str) -> dict:
    """
    Bedanya:
    - Rumus C: clamping sederhana (panjang token) untuk kata tugas/pendek.
    - Rumus B: mapping fonotaktik untuk kata konten/disemvowelling kompleks.
    """
    norm_label = label.upper().strip()
    normalized_token = normalize_typo_token(token)

    # Kasus pronoun gaul: gw/gue/gua dipetakan ke 2 ketuk (vone).
    if normalized_token in {"gw", "gue", "gua"}:
        spoken = "gue" if normalized_token == "gw" else normalized_token
        return {
            "normalized_token": spoken,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(spoken, force_two_syllable=True),
        }

    if norm_label in LABELS_TYPO and len(normalized_token) <= 3:
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token, force_two_syllable=True),
        }

    if len(normalized_token) <= 2 and all(ch not in VOWELS for ch in normalized_token):
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token),
        }

    if _is_content_like_disemvowelling(normalized_token):
        return {
            "normalized_token": normalized_token,
            "rule_used": "B",
            "phoneme": rumus_b_phonotactic_mapping(normalized_token),
        }

    if len(normalized_token) <= 3:
        return {
            "normalized_token": normalized_token,
            "rule_used": "C",
            "phoneme": rumus_c_clamping(normalized_token),
        }

    return {
        "normalized_token": normalized_token,
        "rule_used": "C",
        "phoneme": rumus_c_clamping(normalized_token),
    }


def acoustic_switchboard(token: str, label: str, token_meta: dict | None = None) -> dict:
    """Switch board fonem: jalur A, B/C, ASG, atau O sesuai label IndoBERT."""
    norm = label.upper().strip()
    cleaned = normalize_typo_token(token)

    if norm in LABELS_SINGKATAN:
        ejaan = singkatan_to_ejaan(token)
        return {
            "switch_route": "A_SINGKATAN_G2P_ID",
            "switch_input": ejaan,
            "phoneme": rumus_a_singkatan_simple(token),
        }

    if norm in LABELS_ASING:
        return choose_asg_phoneme(token)

    if norm in LABELS_TYPO:
        typ_gate = should_attempt_typo_recover(token, token_meta=token_meta)

        if typ_gate['attempt_recover']:
            typo_edit_distance = 2 if typ_gate.get('has_reversed_cluster') else 1
            recovered = symspell_recover_typo(
                token,
                allow_no_vowel=False,
                max_edit_distance=typo_edit_distance,
                max_len_delta=2,
            )
            if recovered:
                return {
                    "switch_route": "TYP_SYMSPELL_G2P_ID",
                    "switch_input": recovered['recovered'],
                    "phoneme": g2p_id_phoneme(recovered['recovered']),
                    "bc_info": {
                        "rule_used": "SYM",
                        "normalized_token": recovered['recovered'],
                        "recover": recovered,
                        "gate": typ_gate,
                    },
                }

        if typ_gate['use_bc']:
            bc = rumus_bc_for_dsm_typo(token, norm)
            return {
                "switch_route": f"BC_{bc['rule_used']}",
                "switch_input": bc["normalized_token"],
                "phoneme": bc["phoneme"],
                "bc_info": bc,
            }

        return {
            "switch_route": "TYP_G2P_ID_DIRECT",
            "switch_input": cleaned,
            "phoneme": g2p_id_phoneme(cleaned),
            "bc_info": {
                "rule_used": "DIRECT",
                "normalized_token": cleaned,
                "gate": typ_gate,
            },
        }

    if norm in LABELS_DISEMVOWEL:
        recovered_dsm = dsm_recover_from_skeleton(token)
        if recovered_dsm:
            return {
                "switch_route": "DSM_SKELETON_G2P_ID",
                "switch_input": recovered_dsm['recovered'],
                "phoneme": g2p_id_phoneme(recovered_dsm['recovered']),
                "bc_info": {
                    "rule_used": "DSM_SKEL",
                    "normalized_token": recovered_dsm['recovered'],
                    "recover": recovered_dsm,
                },
            }
        bc = rumus_bc_for_dsm_typo(token, norm)
        return {
            "switch_route": f"BC_{bc['rule_used']}",
            "switch_input": bc["normalized_token"],
            "phoneme": bc["phoneme"],
            "bc_info": bc,
        }

    gate = should_attempt_o_recover(token, token_meta=token_meta)
    if gate['suspicious']:
        recovered_o = symspell_recover_typo(
            token,
            allow_no_vowel=True,
            max_edit_distance=2,
            max_len_delta=2,
        )
        is_strict_confident = recovered_o and recovered_o['distance'] <= 1 and recovered_o['count'] >= 2
        is_transposition_confident = recovered_o and gate.get('has_reversed_cluster') and recovered_o['distance'] <= 2 and recovered_o['count'] >= 1
        if is_strict_confident or is_transposition_confident:
            return {
                "switch_route": "O_SYMSPELL_G2P_ID",
                "switch_input": recovered_o['recovered'],
                "phoneme": g2p_id_phoneme(recovered_o['recovered']),
                "bc_info": {
                    "rule_used": "O_SYM",
                    "normalized_token": recovered_o['recovered'],
                    "recover": recovered_o,
                    "gate": gate,
                },
            }

    return {
        "switch_route": "O_G2P_ID",
        "switch_input": cleaned,
        "phoneme": g2p_id_phoneme(cleaned),
    }

def _decode_bi_to_syllables(units: list[str], bi_pattern: str | None) -> dict:
    if not units:
        return {
            'valid': False,
            'labels': [],
            'syllables_units': [],
            'syllables_text': [],
            'error': 'units_empty',
        }

    labels = list((bi_pattern or '').strip())
    if len(labels) != len(units) or any(lb not in {'B', 'I'} for lb in labels):
        syll_text = ''.join(units)
        return {
            'valid': False,
            'labels': labels,
            'syllables_units': [units],
            'syllables_text': [syll_text],
            'error': 'invalid_bi_pattern',
        }

    syllables_units: list[list[str]] = []
    current: list[str] = []
    for i, (u, lb) in enumerate(zip(units, labels)):
        if i == 0 or lb == 'B':
            if current:
                syllables_units.append(current)
            current = [u]
        else:
            current.append(u)
    if current:
        syllables_units.append(current)

    syllables_text = [''.join(parts) for parts in syllables_units]
    return {
        'valid': True,
        'labels': labels,
        'syllables_units': syllables_units,
        'syllables_text': syllables_text,
        'error': None,
    }


def _is_vowel_unit_for_nucleus(unit: str) -> bool:
    if not unit:
        return False
    if _crf3_is_vowel(unit):
        return True
    if unit in {'eɪ', 'aɪ', 'aʊ', 'ɔɪ', 'oʊ'}:
        return True
    return any(ch in CRF3_VOWELS for ch in unit)


def _repair_asg_syllable_nucleus(decoded: dict) -> tuple[dict, bool]:
    # Rule-based guardrail pasca-CRF agar setiap silabel ASG punya nucleus vokal.
    # Ini aturan fonotaktik, bukan parameter yang dioptimasi numerik.
    syllables_units = [list(parts) for parts in decoded.get('syllables_units') or []]
    if len(syllables_units) <= 1:
        return decoded, False

    def has_nucleus(parts: list[str]) -> bool:
        return any(_is_vowel_unit_for_nucleus(u) for u in parts)

    changed = False
    i = 0
    step_guard = 0
    step_limit = max(16, len(syllables_units) * 8)
    while i < len(syllables_units):
        step_guard += 1
        if step_guard > step_limit:
            break
        if has_nucleus(syllables_units[i]):
            i += 1
            continue

        if len(syllables_units) <= 1:
            break

        left_idx = i - 1 if i > 0 else None
        right_idx = i + 1 if i + 1 < len(syllables_units) else None
        fixed = False

        if right_idx is not None and syllables_units[right_idx]:
            donor = syllables_units[right_idx]
            if has_nucleus(donor):
                first_vowel_idx = next((j for j, u in enumerate(donor) if _is_vowel_unit_for_nucleus(u)), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[1:])
                if first_vowel_idx == 0 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].append(donor.pop(0))
                    changed = True
                    fixed = True

        if (not fixed) and left_idx is not None and syllables_units[left_idx]:
            donor = syllables_units[left_idx]
            if has_nucleus(donor):
                last_vowel_idx = next((j for j in range(len(donor) - 1, -1, -1) if _is_vowel_unit_for_nucleus(donor[j])), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[:-1])
                if last_vowel_idx == len(donor) - 1 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].insert(0, donor.pop())
                    changed = True
                    fixed = True

        if not fixed:
            i += 1
            continue

        i = max(i - 1, 0)

    for j in range(len(syllables_units) - 1):
        left = syllables_units[j]
        right = syllables_units[j + 1]
        if len(left) != 1 or not _is_vowel_unit_for_nucleus(left[0]) or not right:
            continue

        first_vowel_idx = next((k for k, u in enumerate(right) if _is_vowel_unit_for_nucleus(u)), None)
        if first_vowel_idx is None:
            continue

        if first_vowel_idx >= 2:
            left.append(right.pop(0))
            changed = True

    if not changed:
        return decoded, False

    repaired = {
        **decoded,
        'syllables_units': syllables_units,
        'syllables_text': [''.join(parts) for parts in syllables_units],
    }
    return repaired, True


def _repair_syllable_nucleus_merge(decoded: dict) -> tuple[dict, bool]:
    syllables_units = [list(parts) for parts in decoded.get('syllables_units') or []]
    if len(syllables_units) <= 1:
        return decoded, False

    def has_nucleus(parts: list[str]) -> bool:
        return any(_is_vowel_unit_for_nucleus(u) for u in parts)

    changed = False
    i = 0
    step_guard = 0
    step_limit = max(16, len(syllables_units) * 8)
    while i < len(syllables_units):
        step_guard += 1
        if step_guard > step_limit:
            break
        if has_nucleus(syllables_units[i]):
            i += 1
            continue

        if len(syllables_units) <= 1:
            break

        left_idx = i - 1 if i > 0 else None
        right_idx = i + 1 if i + 1 < len(syllables_units) else None
        fixed = False

        if right_idx is not None and syllables_units[right_idx]:
            donor = syllables_units[right_idx]
            if has_nucleus(donor):
                first_vowel_idx = next((j for j, u in enumerate(donor) if _is_vowel_unit_for_nucleus(u)), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[1:])
                if first_vowel_idx == 0 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].append(donor.pop(0))
                    changed = True
                    fixed = True

        if (not fixed) and left_idx is not None and syllables_units[left_idx]:
            donor = syllables_units[left_idx]
            if has_nucleus(donor):
                last_vowel_idx = next((j for j in range(len(donor) - 1, -1, -1) if _is_vowel_unit_for_nucleus(donor[j])), None)
                donor_keeps_nucleus = any(_is_vowel_unit_for_nucleus(u) for u in donor[:-1])
                if last_vowel_idx == len(donor) - 1 and len(donor) >= 2 and donor_keeps_nucleus:
                    syllables_units[i].insert(0, donor.pop())
                    changed = True
                    fixed = True

        if fixed:
            i = max(i - 1, 0)
            continue


        target_idx = None
        if right_idx is not None and has_nucleus(syllables_units[right_idx]):
            target_idx = right_idx
        elif left_idx is not None and has_nucleus(syllables_units[left_idx]):
            target_idx = left_idx
        elif right_idx is not None:
            target_idx = right_idx
        elif left_idx is not None:
            target_idx = left_idx

        if target_idx is None:
            i += 1
            continue

        stranded = syllables_units[i]
        if target_idx < i:
            syllables_units[target_idx].extend(stranded)
            syllables_units.pop(i)
            changed = True
            i = max(target_idx, 0)
            continue

        syllables_units[target_idx] = stranded + syllables_units[target_idx]
        syllables_units.pop(i)
        changed = True

    if not changed:
        return decoded, False

    repaired = {
        **decoded,
        'syllables_units': syllables_units,
        'syllables_text': [''.join(parts) for parts in syllables_units],
    }
    return repaired, True


def _repair_id_vowel_hiatus(decoded: dict, token: str, switch_route: str | None) -> tuple[dict, bool]:
    # Rule khusus Indonesia untuk pola hiatus vokal tertentu (mis. ai/au/oi/ei)
    # agar pemenggalan lebih natural pada kasus umum; tidak berasal dari tuning threshold.
    if switch_route != 'O_G2P_ID':
        return decoded, False

    syllables_units = [list(parts) for parts in decoded.get('syllables_units') or []]
    if len(syllables_units) <= 1:
        return decoded, False

    t = normalize_typo_token(token)
    changed = False

    # Merge pola vokal tunggal terpisah (mis. na | i -> nai pada "menunaikan").
    if any(dg in t for dg in ('ai', 'au', 'oi', 'ei')):
        i = 1
        while i < len(syllables_units) - 1:
            prev = syllables_units[i - 1]
            cur = syllables_units[i]
            nxt = syllables_units[i + 1]
            if (
                len(cur) == 1
                and _is_vowel_unit_for_nucleus(cur[0])
                and prev
                and _is_vowel_unit_for_nucleus(prev[-1])
                and nxt
                and (not _is_vowel_unit_for_nucleus(nxt[0]))
            ):
                prev.extend(cur)
                syllables_units.pop(i)
                changed = True
                continue
            i += 1

    # Split pola ...au|han menjadi ...a|u|han (mis. "berjauhan").
    i = 0
    while i < len(syllables_units) - 1:
        left = syllables_units[i]
        right = syllables_units[i + 1]
        if (
            len(left) >= 2
            and _is_vowel_unit_for_nucleus(left[-1])
            and _is_vowel_unit_for_nucleus(left[-2])
            and right
            and right[0] == 'h'
            and len(right) >= 2
            and _is_vowel_unit_for_nucleus(right[1])
            and 'auh' in t
        ):
            trailing_vowel = left.pop()
            syllables_units.insert(i + 1, [trailing_vowel])
            changed = True
            i += 2
            continue
        i += 1

    if not changed:
        return decoded, False

    repaired = {
        **decoded,
        'syllables_units': syllables_units,
        'syllables_text': [''.join(parts) for parts in syllables_units],
    }
    return repaired, True


def _extract_rhyme_tail(syllable_text: str) -> str:
    if not syllable_text:
        return ''
    chars = [ch for ch in syllable_text if not ch.isspace()]
    if not chars:
        return ''
    for i in range(len(chars) - 1, -1, -1):
        if chars[i] in CRF3_VOWELS:
            return ''.join(chars[i:])
    return ''.join(chars)


def analyze_line_prosody_full(text: str) -> dict:
    token_rows = predict_with_routing(text)
    analyzed_tokens = []
    line_syllables = []
    line_units = []
    line_bi_flat = []

    def _has_nucleus(parts: list[str]) -> bool:
        return any(_is_vowel_unit_for_nucleus(u) for u in parts)

    for row in token_rows:
        token_label = row.get('label')
        if token_label in LABELS_SINGKATAN:
            token_syllables = [u for u in singkatan_to_ejaan(row.get('token') or '').split() if u]
            line_syllables.extend(token_syllables)
            analyzed_tokens.append({
                **row,
                'units': [],
                'unit_count': 0,
                'bi_pattern': '',
                'bi_valid': True,
                'bi_error': None,
                'syllables': token_syllables,
                'syllable_count': len(token_syllables),
                'rhyme_tail_token': _extract_rhyme_tail(token_syllables[-1]) if token_syllables else '',
                'asg_nucleus_fixed': False,
                'syllable_repair': 'SKT_SKIP_POSTPROCESS',
            })
            continue

        phoneme = row.get('phoneme') or ''
        units = phoneme_to_units(phoneme)
        bi_pattern = row.get('bi_pattern')
        if not bi_pattern:
            bi_pattern = predict_bi_from_phoneme(phoneme)

        decoded = _decode_bi_to_syllables(units, bi_pattern)
        asg_nucleus_fixed = False
        syllable_repair = None
        if str(row.get('switch_route', '')).startswith('ASG_'):
            decoded, asg_nucleus_fixed = _repair_asg_syllable_nucleus(decoded)
            if asg_nucleus_fixed:
                syllable_repair = 'asg_nucleus_fix'
            has_stranded = any((not _has_nucleus(parts)) for parts in (decoded.get('syllables_units') or []))
            if has_stranded:
                decoded, merged_after_asg = _repair_syllable_nucleus_merge(decoded)
                if merged_after_asg:
                    syllable_repair = 'asg_nucleus_merge'
        else:
            decoded, generic_nucleus_fixed = _repair_syllable_nucleus_merge(decoded)
            if generic_nucleus_fixed:
                syllable_repair = 'nucleus_merge'
            decoded, id_hiatus_fixed = _repair_id_vowel_hiatus(
                decoded,
                token=row.get('token') or '',
                switch_route=row.get('switch_route'),
            )
            if id_hiatus_fixed:
                syllable_repair = 'id_vowel_hiatus_fix'
        token_syllables = decoded['syllables_text']

        line_syllables.extend(token_syllables)
        line_units.extend(units)
        if decoded['valid']:
            line_bi_flat.extend(decoded['labels'])

        analyzed = {
            **row,
            'units': units,
            'unit_count': len(units),
            'bi_pattern': bi_pattern,
            'bi_valid': decoded['valid'],
            'bi_error': decoded['error'],
            'syllables': token_syllables,
            'syllable_count': len(token_syllables),
            'rhyme_tail_token': _extract_rhyme_tail(token_syllables[-1]) if token_syllables else '',
            'asg_nucleus_fixed': asg_nucleus_fixed,
            'syllable_repair': syllable_repair,
        }
        analyzed_tokens.append(analyzed)

    total_syllables = len(line_syllables)
    final_syllable = line_syllables[-1] if line_syllables else ''
    final_rhyme = _extract_rhyme_tail(final_syllable)

    return {
        'input_text': text,
        'tokens': analyzed_tokens,
        'line_units': line_units,
        'line_unit_count': len(line_units),
        'line_bi_pattern_flat': ''.join(line_bi_flat),
        'line_syllables': line_syllables,
        'line_syllable_count': total_syllables,
        'final_syllable': final_syllable,
        'final_rhyme': final_rhyme,
    }


def print_line_prosody_analysis(report: dict) -> None:
    print('=== ANALISIS LENGKAP LAYER-1 -> LAYER-3 ===')
    print(f"Input text         : {report['input_text']}")
    print(f"Total token        : {len(report['tokens'])}")
    print()

    print('--- Detail per token ---')
    for idx, row in enumerate(report['tokens'], start=1):
        conf = row.get('model_confidence')
        margin = row.get('model_margin')
        conf_txt = f"{conf:.3f}" if conf is not None else '-'
        margin_txt = f"{margin:.3f}" if margin is not None else '-'

        print(
            f"[{idx:02d}] token={row['token']} | raw={row['raw_label']} | final={row['label']} | "
            f"route={row['route']} | switch={row['switch_route']} | conf={conf_txt} margin={margin_txt}"
        )
        if row.get('bc_info'):
            bc = row['bc_info']
            print(f"     bc_rule={bc.get('rule_used')} | normalized={bc.get('normalized_token')}")
        print(f"     phoneme={row.get('phoneme', '')}")
        print(f"     units={row['units']} (n={row['unit_count']})")
        print(
            f"     bi={row.get('bi_pattern') or '-'} | bi_valid={row.get('bi_valid')}"
            + (f" | bi_error={row.get('bi_error')}" if row.get('bi_error') else '')
        )
        print(f"     syllables={row.get('syllables', [])} | syllable_count={row.get('syllable_count', 0)}")
        if row.get('syllable_repair'):
            print(f"     postprocess={row.get('syllable_repair')}")
        print(f"     token_rhyme_tail={row.get('rhyme_tail_token', '')}")

    print('\n--- Agregasi baris ---')
    print(f"line_units          : {report['line_units']}")
    print(f"line_unit_count     : {report['line_unit_count']}")
    print(f"line_bi_pattern     : {report['line_bi_pattern_flat']}")
    print(f"line_syllables      : {report['line_syllables']}")
    print(f"JUMLAH_SILABEL      : {report['line_syllable_count']}")
    print(f"SILABEL_AKHIR       : {report['final_syllable']}")
    print(f"RIMA_AKHIR          : {report['final_rhyme']}")

def _rhyme_key(rhyme_tail: str) -> str:
    t = (rhyme_tail or '').strip()
    if not t:
        return '-'
    if len(t) >= 2:
        return t[-2:]
    return t


def infer_rhyme_scheme(line_reports: list[dict]) -> str:
    keys = [_rhyme_key(rep.get('final_rhyme', '')) for rep in line_reports]
    mapping = {}
    next_label_ord = ord('a')
    scheme = []
    for key in keys:
        if key not in mapping:
            mapping[key] = chr(next_label_ord)
            next_label_ord += 1
        scheme.append(mapping[key])
    return '-'.join(scheme)


def analyze_pantun_multiline(text: str) -> dict:
    lines = [ln.strip() for ln in str(text).splitlines() if ln.strip()]
    line_reports = []
    for i, line in enumerate(lines, start=1):
        rep = analyze_line_prosody_full(line)
        rep['line_no'] = i
        line_reports.append(rep)

    rhyme_scheme = infer_rhyme_scheme(line_reports)
    total_syllables = sum(rep.get('line_syllable_count', 0) for rep in line_reports)

    return {
        'raw_text': text,
        'line_count': len(line_reports),
        'lines': line_reports,
        'total_syllables_all_lines': total_syllables,
        'rhyme_scheme': rhyme_scheme,
    }


def build_pantun_compact_output(report: dict) -> dict:
    lines = report.get('lines', [])

    compact = {
        'skema_rima': report.get('rhyme_scheme', ''),
        'rima_akhir_baris_1': '',
        'rima_akhir_baris_2': '',
        'rima_akhir_baris_3': '',
        'rima_akhir_baris_4': '',
        'suku_kata_baris_1': '',
        'suku_kata_baris_2': '',
        'suku_kata_baris_3': '',
        'suku_kata_baris_4': '',
    }

    for i in range(min(4, len(lines))):
        rep = lines[i]
        compact[f'rima_akhir_baris_{i+1}'] = rep.get('final_rhyme', '')
        compact[f'suku_kata_baris_{i+1}'] = rep.get('line_syllable_count', '')

    return compact


def print_pantun_multiline_analysis(report: dict, verbose: bool = False) -> None:
    compact = build_pantun_compact_output(report)
    cols = [
        'skema_rima',
        'rima_akhir_baris_1',
        'rima_akhir_baris_2',
        'rima_akhir_baris_3',
        'rima_akhir_baris_4',
        'suku_kata_baris_1',
        'suku_kata_baris_2',
        'suku_kata_baris_3',
        'suku_kata_baris_4',
    ]

    print(','.join(str(compact[col]) for col in cols))

    if not verbose:
        return

    print('\n=== ANALISIS PANTUN MULTI-BARIS (VERBOSE) ===')
    print(f"Jumlah baris terdeteksi : {report['line_count']}")
    print(f"Total silabel (semua)   : {report['total_syllables_all_lines']}")
    print(f"Skema rima              : {report['rhyme_scheme']}")

    for line_rep in report['lines']:
        print('\n' + '=' * 64)
        print(f"BARIS {line_rep['line_no']}: {line_rep['input_text']}")
        print_line_prosody_analysis(line_rep)



In [ ]:
# Contoh uji input pantun multi-baris (pisahkan dengan newline "\n")
sample_pantun = """lagi ngeliat ilc sekarang\npetinggi sunda empire bikin ngakak\ncintaku jangan kau buang\naku gamau jantungku berhenti mendadak"""

pantun_report = analyze_pantun_multiline(sample_pantun)

# Default: output ringkas sesuai format dataset
print_pantun_multiline_analysis(pantun_report)

# Verbose (opsional): tampilkan semua detail per token/baris
# print_pantun_multiline_analysis(pantun_report, verbose=True)


In [ ]:
# Ablation / tuning report untuk menjawab reviewer Q1
# Menilai dampak threshold conf/margin/DSM-score pada validation set.

import csv
from itertools import product

VALIDATION_CSV_PATH = '/mnt/d/workspace/penelitian/Disertasi/coding/rima/data/article5/pantun_gaul.csv'
MAX_SAMPLES = None  # isi int (mis. 200) untuk quick run


def _to_int_safe(x):
    try:
        return int(str(x).strip())
    except Exception:
        return None


def _eq_str(a, b):
    return str(a).strip() == str(b).strip()


def _evaluate_current_pipeline_on_validation(csv_path, max_samples=None):
    n_pantun = 0
    line_total = 0
    line_syllable_exact = 0
    line_syllable_abs_err_sum = 0
    line_rhyme_exact = 0
    scheme_exact = 0
    strict_exact = 0

    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = row.get('text_pantun', '')
            if not text:
                continue

            report = analyze_pantun_multiline(text)
            compact = build_pantun_compact_output(report)

            n_pantun += 1
            if max_samples is not None and n_pantun > max_samples:
                break

            row_all_ok = True
            for i in range(1, 5):
                pred_suku = _to_int_safe(compact.get(f'suku_kata_baris_{i}', ''))
                true_suku = _to_int_safe(row.get(f'suku_kata_baris_{i}', ''))
                pred_rima = compact.get(f'rima_akhir_baris_{i}', '')
                true_rima = row.get(f'rima_akhir_baris_{i}', '')

                if pred_suku is not None and true_suku is not None:
                    line_total += 1
                    if pred_suku == true_suku:
                        line_syllable_exact += 1
                    line_syllable_abs_err_sum += abs(pred_suku - true_suku)
                else:
                    row_all_ok = False

                if _eq_str(pred_rima, true_rima):
                    line_rhyme_exact += 1
                else:
                    row_all_ok = False

            if _eq_str(compact.get('skema_rima', ''), row.get('skema_rima', '')):
                scheme_exact += 1
            else:
                row_all_ok = False

            if row_all_ok:
                strict_exact += 1

    if n_pantun == 0 or line_total == 0:
        return None

    return {
        'n_pantun': n_pantun,
        'line_syllable_acc': line_syllable_exact / line_total,
        'line_syllable_mae': line_syllable_abs_err_sum / line_total,
        'line_rhyme_acc': line_rhyme_exact / line_total,
        'scheme_acc': scheme_exact / n_pantun,
        'strict_acc': strict_exact / n_pantun,
    }


def _make_is_low_confidence_label(conf_thr, margin_thr):
    def _fn(token_meta, conf_threshold=conf_thr, margin_threshold=margin_thr):
        if not token_meta:
            return False
        confidence = token_meta.get('confidence')
        margin = token_meta.get('margin')
        if confidence is None or margin is None:
            return False
        return float(confidence) <= conf_threshold or float(margin) <= margin_threshold
    return _fn


def _make_is_likely_disemvowelling(score_threshold):
    def _fn(token):
        t = normalize_typo_token(token)
        if len(t) < 3:
            return False

        vowels = set('aiueo')
        vowel_count = sum(ch in vowels for ch in t)
        consonant_ratio = (len(t) - vowel_count) / len(t)

        max_cons_run = 0
        curr_run = 0
        for ch in t:
            if ch in vowels:
                curr_run = 0
            else:
                curr_run += 1
                max_cons_run = max(max_cons_run, curr_run)

        consonant_clusters = ('ng', 'ny', 'kh', 'sy', 'tr', 'br', 'gr', 'pr', 'kr', 'sk', 'st', 'nd', 'mb')
        cluster_hits = sum(1 for c in consonant_clusters if c in t)

        score = 0
        if vowel_count == 0:
            score += 3
        elif vowel_count == 1 and len(t) >= 4:
            score += 2
        if consonant_ratio >= 0.70:
            score += 2
        if max_cons_run >= 3:
            score += 2
        if cluster_hits >= 1:
            score += 1
        if len(t) >= 4 and t[0] not in vowels and t[-1] not in vowels and vowel_count <= 1:
            score += 1

        return score >= score_threshold
    return _fn


def run_threshold_ablation_report(
    csv_path=VALIDATION_CSV_PATH,
    conf_grid=(0.75, 0.80, 0.85),
    margin_grid=(0.35, 0.45, 0.55),
    dsm_score_grid=(3, 4, 5),
    max_samples=MAX_SAMPLES,
):
    original_low_conf = is_low_confidence_label
    original_dsm_fn = is_likely_disemvowelling

    rows = []
    try:
        for conf_thr, margin_thr, dsm_thr in product(conf_grid, margin_grid, dsm_score_grid):
            globals()['is_low_confidence_label'] = _make_is_low_confidence_label(conf_thr, margin_thr)
            globals()['is_likely_disemvowelling'] = _make_is_likely_disemvowelling(dsm_thr)

            metrics = _evaluate_current_pipeline_on_validation(csv_path, max_samples=max_samples)
            if metrics is None:
                continue
            rows.append({
                'conf_thr': conf_thr,
                'margin_thr': margin_thr,
                'dsm_score_thr': dsm_thr,
                **metrics,
            })
    finally:
        globals()['is_low_confidence_label'] = original_low_conf
        globals()['is_likely_disemvowelling'] = original_dsm_fn

    if not rows:
        print('Tidak ada hasil. Cek path dataset / fungsi evaluasi.')
        return []

    rows_sorted = sorted(
        rows,
        key=lambda x: (x['strict_acc'], x['scheme_acc'], x['line_rhyme_acc'], x['line_syllable_acc'], -x['line_syllable_mae']),
        reverse=True,
    )

    print('=== TOP 10 THRESHOLD COMBINATIONS ===')
    for i, r in enumerate(rows_sorted[:10], start=1):
        print(
            f"{i:02d}. conf={r['conf_thr']:.2f} margin={r['margin_thr']:.2f} dsm={r['dsm_score_thr']} | "
            f"strict={r['strict_acc']:.4f} scheme={r['scheme_acc']:.4f} rhyme={r['line_rhyme_acc']:.4f} "
            f"syl_acc={r['line_syllable_acc']:.4f} syl_mae={r['line_syllable_mae']:.4f} n={r['n_pantun']}"
        )

    return rows_sorted


# ON/OFF: default False agar aman saat Run All
RUN_ABLATION_REPORT = False
ablation_rows = []
if RUN_ABLATION_REPORT:
    # Jalankan report (hasil tersimpan di ablation_rows)
    ablation_rows = run_threshold_ablation_report()
else:
    print('Ablation report dilewati (RUN_ABLATION_REPORT=False). Ubah ke True untuk menjalankan.')

